# Saudi Plate OCR — REAL-ONLY training (clean + auto-save)

Trains the recognizer **only** on the real Kaggle plates (`saudi-license-plate-characters`),
and **zips the model automatically** the moment training finishes so it can't be lost.

**Before you run (once):**
1. Session options (right panel) -> Accelerator: **GPU T4 x2**, Internet: **On**.
2. **Add-ons -> Secrets** -> add a secret named **GH_TOKEN** = your GitHub token, and toggle it ON.
3. **+ Add Input** -> search **saudi-license-plate-characters** -> Add.

Then run the 4 cells below top to bottom. Cell 4 is the long one (~15-20 min); when it prints
**ZIP READY**, run Cell 5 and click the link to download the model to your computer.

## 1. Install

In [ ]:
!apt-get -qq update && apt-get -qq -y install fonts-noto-core fonts-kacst fonts-hosny-amiri fonts-dejavu-core > /dev/null 2>&1
!pip install -q paddlepaddle-gpu==2.6.2 --extra-index-url https://www.paddlepaddle.org.cn/packages/stable/cu118/
!rm -rf PaddleOCR
!git clone -q --depth 1 -b release/2.7 https://github.com/PaddlePaddle/PaddleOCR.git
!pip install -q -r PaddleOCR/requirements.txt
!pip install -q 'paddle2onnx==1.1.0' onnxruntime
!pip install -q 'numpy<2'
import paddle
print('paddle', paddle.__version__, '| GPU:', paddle.device.is_compiled_with_cuda())
assert paddle.device.is_compiled_with_cuda(), 'No GPU - set Accelerator to GPU in Session options'

## 2. Get our code from GitHub

In [ ]:
from kaggle_secrets import UserSecretsClient
import subprocess, os
TOKEN = UserSecretsClient().get_secret('GH_TOKEN').strip()
subprocess.run('rm -rf saudi-plate-ocr', shell=True)
subprocess.run(['git','clone','-q',f'https://{TOKEN}@github.com/Tanzimalam1454/saudi-plate-ocr.git'])
for f in ['plate_spec.py','prepare_char_dataset.py','saudi_rec_config.yml','saudi_rec_infer.py']:
    subprocess.run(['cp', f'saudi-plate-ocr/{f}', '.'])
assert all(os.path.exists(f) for f in ['plate_spec.py','prepare_char_dataset.py','saudi_rec_config.yml'])
print('files in place')

## 3. Download the base recognizer (Arabic PP-OCRv4)

In [ ]:
import os, glob
if not glob.glob('pretrain/arabic_PP-OCRv*_rec_train'):
    os.makedirs('pretrain', exist_ok=True)
    !cd pretrain && wget -q https://paddleocr.bj.bcebos.com/PP-OCRv4/multilingual/arabic_PP-OCRv4_rec_train.tar && tar xf arabic_PP-OCRv4_rec_train.tar
!ls pretrain

## 4. Prepare real crops, train REAL-ONLY, and auto-save (~15-20 min)

Finds the Kaggle dataset, turns all 593 plates into ~978 real labeled half-crops, trains only on them,
then zips the trained model automatically. Watch the eval `acc` for real-world accuracy.

In [ ]:
import os, glob, yaml, shutil

# character dictionary (from plate_spec)
os.makedirs('dataset', exist_ok=True)
if not os.path.exists('dataset/saudi_plate_dict.txt'):
    from plate_spec import dictionary_chars
    open('dataset/saudi_plate_dict.txt','w',encoding='utf-8').write("\n".join(dictionary_chars())+"\n")

# find the Kaggle dataset (folder that holds train/ and test/) and build real crops
SRC=None
for root,dirs,files in os.walk('/kaggle/input'):
    low=[d.lower() for d in dirs]
    if 'train' in low and 'test' in low and glob.glob(os.path.join(root,'**','*.xml'),recursive=True):
        SRC=root; break
if SRC is None:
    hits=glob.glob('/kaggle/input/**/*.xml', recursive=True)
    assert hits, "No dataset found - use + Add Input to add 'saudi-license-plate-characters' first!"
    SRC=os.path.dirname(hits[0])
print('dataset:', SRC)
os.system(f'python prepare_char_dataset.py --src "{SRC}" --out dataset --val-frac 0.2')
n=sum(1 for l in open('dataset/char_train_label.txt') if l.strip())
print('real train crops:', n)

# REAL-ONLY fine-tune config (from the base Arabic model)
cfg=yaml.safe_load(open('saudi_rec_config.yml'))
cfg['Global']['pretrained_model']='./pretrain/arabic_PP-OCRv4_rec_train/best_accuracy'
cfg['Global']['checkpoints']=None
cfg['Global']['save_model_dir']='./output/saudi_rec_chars'
cfg['Global']['epoch_num']=60
cfg['Optimizer']['lr']['learning_rate']=0.0005
cfg['Optimizer']['lr']['warmup_epoch']=2
cfg['Train']['dataset']['label_file_list']=['./dataset/char_train_label.txt']
cfg['Eval']['dataset']['label_file_list']=['./dataset/char_val_label.txt']
cfg['Train']['loader']['batch_size_per_card']=32
cfg['Global']['eval_batch_step']=[0, max(1, n//32)]
yaml.safe_dump(cfg, open('saudi_rec_real.yml','w'), allow_unicode=True, sort_keys=False)

# train
os.system('python PaddleOCR/tools/train.py -c saudi_rec_real.yml')

# AUTO-SAVE: zip the trained model immediately so a session timeout can't lose it
dst='/kaggle/working/models_to_download'; shutil.rmtree(dst, ignore_errors=True); os.makedirs(dst)
got=glob.glob('/kaggle/working/output/saudi_rec_chars/best_accuracy*')
for f in got: shutil.copy(f, dst)
for f in ['dataset/saudi_plate_dict.txt','saudi_rec_real.yml']:
    if os.path.exists(f): shutil.copy(f, dst)
shutil.make_archive('/kaggle/working/saudi_ocr_model','zip',dst)
if got:
    print('ZIP READY:', round(os.path.getsize('/kaggle/working/saudi_ocr_model.zip')/1e6,1),'MB  -> run the next cell to download')
else:
    print('WARNING: no model files found - training may have failed; scroll up for errors')

## 5. Download the model

In [ ]:
import os; os.chdir('/kaggle/working')
from IPython.display import FileLink
FileLink('saudi_ocr_model.zip')